# The Analogous Learnings Daily Briefer

This is my Week 8 Capstone Project. It autonomously scans tech news, uses a Modal-deployed RAG pipeline to explain complex tech concepts using Football analogies, and then texts that analogy to my phone using Pushover.

In [15]:
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from week8.community_contributions.toluwalemi.agents.scanner import ScannerAgent
from week8.community_contributions.toluwalemi.agents.analyzer import AnalyzerAgent
from week8.community_contributions.toluwalemi.agents.messenger import MessengerAgent

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4o-mini"

## Initialize Agents and Tools

In [16]:
scanner = ScannerAgent(count=3)
analyzer = AnalyzerAgent()
messenger = MessengerAgent()

def scan_tech_news() -> str:
    return scanner.scan()

def explain_with_football(tech_concept: str) -> str:
    return analyzer.analyze(tech_concept)

def notify_user(title: str, analogy_explanation: str) -> str:
    return messenger.push(title, analogy_explanation)


In [17]:
scan_tool = {
    "name": "scan_tech_news",
    "description": "Returns top tech news articles scraped from the internet",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

explain_tool = {
    "name": "explain_with_football",
    "description": "Given a complex tech concept or news headline, use a RAG pipeline to explain it using a grounded football analogy",
    "parameters": {
        "type": "object",
        "properties": {
            "tech_concept": {
                "type": "string",
                "description": "The tech headline or concept to be explained"
            }
        },
        "required": ["tech_concept"],
        "additionalProperties": False
    }
}

notify_tool = {
    "name": "notify_user",
    "description": "Send the user a push notification summarizing the tech concept via the football analogy",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "A snappy title combining the tech concept and football"
            },
            "analogy_explanation": {
                "type": "string",
                "description": "The generated explanation of the tech concept via football"
            }
        },
        "required": ["title", "analogy_explanation"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": scan_tool},
    {"type": "function", "function": explain_tool},
    {"type": "function", "function": notify_tool}
]

## The Orchestrator

Run until all tools execute.

In [18]:
TOOL_DISPATCH = {
    "scan_tech_news": scan_tech_news,
    "explain_with_football": explain_with_football,
    "notify_user": notify_user,
}

def handle_tool_call(message, status_update):
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        status_update(f"Executing Tool: {tool_name} with args {str(arguments)[:50]}...")

        tool = TOOL_DISPATCH.get(tool_name)
        if tool is None:
            result = f"Error: Unknown tool '{tool_name}'"
        else:
            result = tool(**arguments)

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

def run_autonomous_workflow(log_callback):
    system_message = "You are an autonomous orchestrator. You find the most complex-sounding tech news article, use your tools to explain it with a football analogy, and push the notification. Return OK when finished."
    user_message = "Start the complete autonomous workflow now."

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    log_callback("Pipeline Started: LLM planning...")

    max_iterations = 10
    output_log = ""
    for _ in range(max_iterations):
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

        if response.choices[0].finish_reason == "tool_calls":
            message = response.choices[0].message
            results = handle_tool_call(message, log_callback)
            messages.append(message)
            messages.extend(results)
            output_log += f"Tool returned: {str(results)[:100]}...\n"
        else:
            output_log += f"\nFinal Result: {response.choices[0].message.content}"
            log_callback("Pipeline Finished!")
            break
    else:
        output_log += "\nWarning: Reached maximum iterations without completing."
        log_callback("Pipeline stopped: max iterations reached.")

    return output_log

## The UI Dashboard

In [19]:
def gradio_run():
    def execute_btn():
        yield "Starting AI sequence..."

        log_entries = []
        def log_cb(msg):
            log_entries.append(msg)

        final_output = run_autonomous_workflow(log_cb)
        yield "\n".join(log_entries) + "\n\n---\n" + final_output

    with gr.Blocks(title="Agentic Learnings Updater") as ui:
        gr.Markdown('<div style="text-align: center;font-size:24px">Analogous Learnings Daily Briefer</div>')
        gr.Markdown('<div style="text-align: center">Autonomous agent framework that scans news, translates concepts using RAG on Modal, and alerts you via Pushover.</div>')

        with gr.Row():
            run_btn = gr.Button("Trigger Autonomous Daily Briefing", variant="primary")

        with gr.Row():
            output_console = gr.Textbox(label="Orchestration Log", lines=15, interactive=False)

        run_btn.click(execute_btn, inputs=[], outputs=[output_console])

    ui.launch(inbrowser=True)

In [20]:
gradio_run()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
